# Ecommerce P&L Dashboard
Queries MySQL using SQLAlchemy, builds interactive charts with Plotly, and exports a self-contained HTML file.

In [8]:
# ── Cell 1: Imports & Connection ─────────────────────────────────────────────
import pandas as pd
from sqlalchemy import create_engine

# Update with your credentials
engine = create_engine('mysql+pymysql://root@localhost/ecommercedatabase')

print('Connected successfully')

Connected successfully


In [9]:
# ── Cell 2: Queries → DataFrames ─────────────────────────────────────────────

# Monthly revenue
df_monthly = pd.read_sql("""
    SELECT
        DATE_FORMAT(order_date, '%%Y-%%m') AS month,
        ROUND(SUM(total_amount), 2)      AS revenue,
        COUNT(*)                         AS order_count
    FROM purchase_history
    GROUP BY month
    ORDER BY month
""", engine)

# Estimated P&L — assumes 60% COGS, 15% operating costs
df_monthly['cogs']             = (df_monthly['revenue'] * 0.60).round(2)
df_monthly['gross_profit']     = (df_monthly['revenue'] - df_monthly['cogs']).round(2)
df_monthly['operating_costs']  = (df_monthly['revenue'] * 0.15).round(2)
df_monthly['net_profit']       = (df_monthly['gross_profit'] - df_monthly['operating_costs']).round(2)
df_monthly['gross_margin_pct'] = ((df_monthly['gross_profit'] / df_monthly['revenue']) * 100).round(1)

# Revenue by payment method
df_payment = pd.read_sql("""
    SELECT
        payment_method,
        COUNT(*)                     AS order_count,
        ROUND(SUM(total_amount), 2)  AS total_revenue
    FROM purchase_history
    GROUP BY payment_method
    ORDER BY total_revenue DESC
""", engine)

# Revenue by segment
df_seg_rev = pd.read_sql("""
    SELECT
        cs.segment_name,
        ROUND(SUM(ph.total_amount), 2) AS revenue
    FROM purchase_history ph
    JOIN customer_profiles cp ON ph.user_id = cp.user_id
    JOIN customer_segments cs ON cp.segment_id = cs.segment_id
    GROUP BY cs.segment_name
    ORDER BY revenue DESC
""", engine)

print('Queries complete')
print(df_monthly[['month','revenue','gross_profit','net_profit']])

Queries complete
      month  revenue  gross_profit  net_profit
0   2023-06    85.00         34.00       21.25
1   2023-08    75.00         30.00       18.75
2   2023-12    55.00         22.00       13.75
3   2024-01    89.99         36.00       22.50
4   2024-05   288.00        115.20       72.00
5   2024-06   369.00        147.60       92.25
6   2024-07   155.00         62.00       38.75
7   2024-08   110.00         44.00       27.50
8   2024-09  1208.00        483.20      302.00
9   2024-10  2541.50       1016.60      635.38
10  2024-11  2456.98        982.79      614.24


In [10]:
# ── Cell 3: Build Charts & Export HTML ───────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go

colors = ['#2563EB', '#16A34A', '#CA8A04', '#DC2626']

# Chart 1 — Monthly revenue vs net profit (line)
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=df_monthly['month'], y=df_monthly['revenue'],
    name='Revenue', line=dict(color='#2563EB', width=3), mode='lines+markers'
))
fig1.add_trace(go.Scatter(
    x=df_monthly['month'], y=df_monthly['gross_profit'],
    name='Gross Profit', line=dict(color='#16A34A', width=3), mode='lines+markers'
))
fig1.add_trace(go.Scatter(
    x=df_monthly['month'], y=df_monthly['net_profit'],
    name='Net Profit', line=dict(color='#CA8A04', width=3), mode='lines+markers'
))
fig1.update_layout(title='Monthly Revenue, Gross Profit & Net Profit',
                   xaxis_title='Month', yaxis_title='Amount ($)')

# Chart 2 — Gross margin % over time (bar)
fig2 = px.bar(
    df_monthly, x='month', y='gross_margin_pct',
    title='Gross Margin % by Month',
    labels={'month': 'Month', 'gross_margin_pct': 'Gross Margin (%)'},
    color_discrete_sequence=['#2563EB'],
    text='gross_margin_pct'
)
fig2.update_traces(texttemplate='%{text}%', textposition='outside')

# Chart 3 — Revenue by payment method (pie)
fig3 = px.pie(
    df_payment, names='payment_method', values='total_revenue',
    title='Revenue by Payment Method',
    color_discrete_sequence=['#2563EB','#16A34A','#CA8A04']
)

# Chart 4 — Revenue by segment (bar)
fig4 = px.bar(
    df_seg_rev, x='segment_name', y='revenue',
    title='Revenue by Customer Segment',
    labels={'segment_name': 'Segment', 'revenue': 'Revenue ($)'},
    color='segment_name',
    color_discrete_sequence=['#2563EB','#16A34A','#CA8A04','#DC2626','#9333EA','#0891B2']
)
fig4.update_layout(showlegend=False)

# ── Combine into one HTML file ───────────────────────────────────────────────
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset='utf-8'>
    <title>Ecommerce P&L Dashboard</title>
    <style>
        body {{ font-family: Arial, sans-serif; background: #f8fafc; margin: 0; padding: 20px; }}
        h1 {{ color: #1e293b; border-bottom: 3px solid #2563EB; padding-bottom: 10px; }}
        p {{ color: #475569; font-size: 15px; }}
        .grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-top: 20px; }}
        .card {{ background: white; border-radius: 8px; padding: 10px; box-shadow: 0 1px 4px rgba(0,0,0,0.1); }}
        .full {{ grid-column: span 2; }}
    </style>
</head>
<body>
    <h1>Ecommerce P&L Dashboard</h1>
    <p>Tracks monthly revenue, gross profit, and net profit. Assumes 60% COGS and 15% operating costs. Built with MySQL + Python.</p>
    <div class='grid'>
        <div class='card full'>{fig1.to_html(full_html=False, include_plotlyjs='cdn')}</div>
        <div class='card'>{fig2.to_html(full_html=False, include_plotlyjs=False)}</div>
        <div class='card'>{fig3.to_html(full_html=False, include_plotlyjs=False)}</div>
        <div class='card full'>{fig4.to_html(full_html=False, include_plotlyjs=False)}</div>
    </div>
</body>
</html>
"""

import os

output_path = os.path.expanduser('~/Downloads/pl.html')

with open(output_path, 'w') as f:
    f.write(html_content)

print(f'Dashboard saved → {output_path}')

Dashboard saved → /Users/chenyuzhai/Downloads/pl.html
